In [11]:
!wget https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/train.en

--2026-03-16 15:39:09--  https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/train.en
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15438492 (15M) [text/plain]
Saving to: ‘train.en.2’

train.en.2          100%[===================>]  14.72M  --.-KB/s    in 0.1s    

2026-03-16 15:39:10 (125 MB/s) - ‘train.en.2’ saved [15438492/15438492]



In [12]:
!wget https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/train.de

--2026-03-16 15:39:10--  https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/train.de
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17243198 (16M) [text/plain]
Saving to: ‘train.de.1’

train.de.1          100%[===================>]  16.44M  --.-KB/s    in 0.1s    

2026-03-16 15:39:11 (118 MB/s) - ‘train.de.1’ saved [17243198/17243198]



In [13]:
!wget https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/valid.en
!wget https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/valid.de
!wget https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/test.en
!wget https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/test.de

--2026-03-16 15:39:11--  https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/valid.en
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 698449 (682K) [text/plain]
Saving to: ‘valid.en.1’

valid.en.1          100%[===================>] 682.08K  --.-KB/s    in 0.05s   

2026-03-16 15:39:12 (13.6 MB/s) - ‘valid.en.1’ saved [698449/698449]

--2026-03-16 15:39:12--  https://raw.githubusercontent.com/Deoxysis/Machine-Translation-Using-Transformers/refs/heads/main/valid.de
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting 

In [14]:
def read_split(split, lang):
    file_split = "valid" if split == "val" else split
    with open(f"{file_split}.{lang}", "r", encoding="utf-8") as f:
        return f.read()

train_en = read_split("train", "en")
val_en = read_split("val", "en")
test_en = read_split("test", "en")

print("train.en length in characters:", len(train_en))
print("val.en length in characters:", len(val_en))
print("test.en length in characters:", len(test_en))

train.en length in characters: 15437530
val.en length in characters: 698422
test.en length in characters: 618381


In [15]:
print("train.en sample:", train_en[:130])
print("val.en sample:", val_en[:130])
print("test.en sample:", test_en[:130])

train.en sample: and it can be a very complicated thing, what human health is.
and bringing those two together might seem a very daunting task, but
val.en sample: it's that pyramid.
in mother's milk.
it had two-to-three-to-400 times the toxic loads ever allowed by the epa.
often what jams us 
test.en sample: you know, one of the intense pleasures of travel and one of the delights of ethnographic research is the opportunity to live among


In [16]:
train_de = read_split("train", "de")
val_de = read_split("val", "de")
test_de = read_split("test", "de")

print("train.de length in characters:", len(train_de))
print("val.de length in characters:", len(val_de))
print("test.de length in characters:", len(test_de))

train.de length in characters: 17019583
val.de length in characters: 768790
test.de length in characters: 687735


In [17]:
print("train.de sample:", train_de[:130])
print("val.de sample:", val_de[:130])
print("test.de sample:", test_de[:130])

train.de sample: und was menschliche gesundheit ist, kann auch ziemlich kompliziert sein.
und diese zwei zusammen zu bringen, erscheint vielleicht 
val.de sample: es ist diese pyramide.
durch die muttermilch.
es enthielt das zwei-, drei-, bis 400-fache des grenzwerts an schadstoffen der laut 
test.de sample: wissen sie, eines der großen vernügen beim reisen und eine der freuden bei der ethnographischen forschung ist, gemeinsam mit den m


## MT evolution in PyTorch: from recurrence to attention

### Model ideas
| Model | Idea |
|---|---|
| Seq2Seq LSTM | baseline |
| LSTM + Attention | improvement |
| Transformer | modern approach |

Goal: compare how these architectures process the same source/target batch and understand why attention replaced recurrence.

In [18]:
# Toy MT batch (token IDs) so we can compare architectures without full training.
if "torch" not in globals():
    import torch

if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH = 32
SRC_LEN = 40
TGT_LEN = 40
VOCAB_SIZE = 12000
D_MODEL = 256
N_LAYERS = 2

src = torch.randint(2, VOCAB_SIZE, (BATCH, SRC_LEN), device=device)
tgt = torch.randint(2, VOCAB_SIZE, (BATCH, TGT_LEN), device=device)

print("src shape:", tuple(src.shape), "tgt shape:", tuple(tgt.shape))

src shape: (32, 40) tgt shape: (32, 40)


In [19]:
import math
import time
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
print("PyTorch:", torch.__version__)
print("Device:", device)

class Seq2SeqLSTM(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=2, dropout=0.1):
        super().__init__()
        self.src_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_emb = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.LSTM(d_model, d_model, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.decoder = nn.LSTM(d_model, d_model, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src_e = self.src_emb(src)
        _, (h, c) = self.encoder(src_e)
        tgt_e = self.tgt_emb(tgt)
        dec_out, _ = self.decoder(tgt_e, (h, c))
        return self.out(dec_out)

class LSTMAttentionMT(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.src_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_emb = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.LSTM(d_model, d_model, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.decoder = nn.LSTM(d_model, d_model, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.attn = nn.MultiheadAttention(d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.fuse = nn.Linear(2 * d_model, d_model)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        enc_in = self.src_emb(src)
        enc_out, (h, c) = self.encoder(enc_in)
        dec_in = self.tgt_emb(tgt)
        dec_out, _ = self.decoder(dec_in, (h, c))
        ctx, _ = self.attn(dec_out, enc_out, enc_out)
        fused = torch.tanh(self.fuse(torch.cat([dec_out, ctx], dim=-1)))
        return self.out(fused)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]

class TransformerMT(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=2, n_heads=8, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_emb = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        self.tf = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src_e = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        tgt_e = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=tgt.device)
        h = self.tf(src_e, tgt_e, tgt_mask=tgt_mask)
        return self.out(h)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def benchmark_forward(model, src, tgt, warmup=1, runs=5):
    model.eval()
    for _ in range(warmup):
        _ = model(src, tgt)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        _ = model(src, tgt)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / runs

PyTorch: 2.10.0+cpu
Device: cpu


### Why attention replaced recurrence

1. Parallelism: LSTMs process tokens sequentially, while Transformer self-attention processes all positions in parallel, which is much faster on GPUs.
2. Long-range dependencies: recurrent signals are repeatedly compressed through hidden states; attention directly links any two positions in one step.
3. Path length: for token interactions, recurrence has path length $O(n)$ while self-attention has effective path length $O(1)$ per layer.
4. Quality scaling: attention-based models generally improve more smoothly as data and model size grow.

In practice, LSTM + Attention improved classic Seq2Seq by giving direct access to encoder states, and Transformer generalized this idea by removing recurrence entirely.

Note: this notebook benchmark is a small forward-pass test on CPU, so Transformer can appear slower here. The main advantage of attention shows up strongly in large-scale training/inference on accelerators (GPU/TPU).

In [20]:
models = {
    "Seq2Seq LSTM": Seq2SeqLSTM(VOCAB_SIZE, D_MODEL, N_LAYERS).to(device),
    "LSTM + Attention": LSTMAttentionMT(VOCAB_SIZE, D_MODEL, N_LAYERS).to(device),
    "Transformer": TransformerMT(VOCAB_SIZE, D_MODEL, N_LAYERS).to(device),
}

results = []
for name, model in models.items():
    logits = model(src, tgt)
    avg_s = benchmark_forward(model, src, tgt, warmup=1, runs=5)
    results.append((name, tuple(logits.shape), count_params(model), avg_s))

print(f"{'Model':<20} {'Output shape':<20} {'Params':>12} {'Avg forward (s)':>16}")
print("-" * 74)
for name, shape, params, avg_s in results:
    print(f"{name:<20} {str(shape):<20} {params:>12,} {avg_s:>16.6f}")

Model                Output shape               Params  Avg forward (s)
--------------------------------------------------------------------------
Seq2Seq LSTM         (32, 40, 12000)        11,333,344         0.354037
LSTM + Attention     (32, 40, 12000)        11,727,840         0.304413
Transformer          (32, 40, 12000)        12,915,424         0.310440


## Efficiency vs Performance on the given data

This section uses your loaded train/val splits (de->en) and compares:
- Efficiency: parameters, step time, tokens/sec
- Performance: training loss trend and validation loss (next-token objective)

Interpretation guide:
- If one model has lower val loss but lower tok/s, it is trading efficiency for performance.
- If one model has much higher tok/s with similar val loss, it is the better efficiency-performance balance for this setup.
- On CPU and very short runs, RNN variants can look efficient; on larger GPU training, Transformer typically scales better.

In [21]:
from collections import Counter
import random

if "Seq2SeqLSTM" not in globals():
    raise RuntimeError("Run the PyTorch model-definition cell before this section.")

random.seed(7)
torch.manual_seed(7)

def nonempty_lines(text, max_lines):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[:max_lines]

# Use manageable subsets from the given data for quick apples-to-apples comparison.
train_src_lines = nonempty_lines(train_de, 3000)
train_tgt_lines = nonempty_lines(train_en, 3000)
val_src_lines = nonempty_lines(val_de, 800)
val_tgt_lines = nonempty_lines(val_en, 800)

train_pairs = list(zip(train_src_lines, train_tgt_lines))
val_pairs = list(zip(val_src_lines, val_tgt_lines))

print("train pairs:", len(train_pairs), "val pairs:", len(val_pairs))

def tok(s):
    return s.lower().split()

SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

counter = Counter()
for src_line, tgt_line in train_pairs:
    counter.update(tok(src_line))
    counter.update(tok(tgt_line))

MAX_VOCAB = 12000
vocab_words = [w for w, _ in counter.most_common(MAX_VOCAB - len(SPECIALS))]
itos = SPECIALS + vocab_words
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_EFF = len(itos)

MAX_SRC_LEN = 24
MAX_TGT_LEN = 24

def encode_sentence(text, max_len):
    ids = [stoi.get(w, UNK_ID) for w in tok(text)]
    ids = [BOS_ID] + ids[: max_len - 2] + [EOS_ID]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids

def build_tensors(pairs):
    src_ids = [encode_sentence(src_line, MAX_SRC_LEN) for src_line, _ in pairs]
    tgt_ids = [encode_sentence(tgt_line, MAX_TGT_LEN) for _, tgt_line in pairs]
    src = torch.tensor(src_ids, dtype=torch.long, device=device)
    tgt = torch.tensor(tgt_ids, dtype=torch.long, device=device)
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    return src, tgt_in, tgt_out

train_src, train_tgt_in, train_tgt_out = build_tensors(train_pairs)
val_src, val_tgt_in, val_tgt_out = build_tensors(val_pairs)

print("train tensor shapes:", tuple(train_src.shape), tuple(train_tgt_in.shape), tuple(train_tgt_out.shape))
print("val tensor shapes:", tuple(val_src.shape), tuple(val_tgt_in.shape), tuple(val_tgt_out.shape))

train pairs: 3000 val pairs: 800
train tensor shapes: (3000, 24) (3000, 23) (3000, 23)
val tensor shapes: (800, 24) (800, 23) (800, 23)


In [22]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def sample_batch(src, tgt_in, tgt_out, batch_size):
    n = src.size(0)
    idx = torch.randint(0, n, (batch_size,), device=src.device)
    return src[idx], tgt_in[idx], tgt_out[idx]

def evaluate_loss(model, src, tgt_in, tgt_out, batch_size=64, max_batches=8):
    model.eval()
    losses = []
    with torch.no_grad():
        batches = min(max_batches, max(1, src.size(0) // batch_size))
        for _ in range(batches):
            s, ti, to = sample_batch(src, tgt_in, tgt_out, batch_size)
            logits = model(s, ti)
            loss = criterion(logits.reshape(-1, logits.size(-1)), to.reshape(-1))
            losses.append(loss.item())
    return sum(losses) / len(losses)

def train_steps(model, src, tgt_in, tgt_out, steps=8, batch_size=32, lr=3e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []
    total_tokens = steps * batch_size * tgt_in.size(1)

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    for _ in range(steps):
        s, ti, to = sample_batch(src, tgt_in, tgt_out, batch_size)
        opt.zero_grad()
        logits = model(s, ti)
        loss = criterion(logits.reshape(-1, logits.size(-1)), to.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return {
        "train_loss_last": losses[-1],
        "train_loss_avg": sum(losses) / len(losses),
        "seconds": elapsed,
        "tok_per_sec": total_tokens / max(elapsed, 1e-8),
    }

eff_models = {
    "Seq2Seq LSTM": Seq2SeqLSTM(VOCAB_EFF, d_model=128, n_layers=2).to(device),
    "LSTM + Attention": LSTMAttentionMT(VOCAB_EFF, d_model=128, n_layers=2, n_heads=4).to(device),
    "Transformer": TransformerMT(VOCAB_EFF, d_model=128, n_layers=2, n_heads=4, ff_dim=512).to(device),
}

rows = []
for model_name, mt_model in eff_models.items():
    train_stats = train_steps(mt_model, train_src, train_tgt_in, train_tgt_out, steps=8, batch_size=32, lr=3e-4)
    val_loss = evaluate_loss(mt_model, val_src, val_tgt_in, val_tgt_out, batch_size=64, max_batches=8)
    rows.append(
        {
            "model": model_name,
            "params": count_params(mt_model),
            "train_loss_avg": train_stats["train_loss_avg"],
            "val_loss": val_loss,
            "seconds": train_stats["seconds"],
            "tok_per_sec": train_stats["tok_per_sec"],
        }
    )

print(f"{'Model':<20} {'Params':>10} {'TrainLoss':>11} {'ValLoss':>10} {'Time(s)':>9} {'Tok/s':>11}")
print("-" * 80)
for r in rows:
    print(
        f"{r['model']:<20} {r['params']:>10,} {r['train_loss_avg']:>11.4f} "
        f"{r['val_loss']:>10.4f} {r['seconds']:>9.3f} {r['tok_per_sec']:>11.1f}"
    )

best_val = min(rows, key=lambda x: x["val_loss"])
fastest = max(rows, key=lambda x: x["tok_per_sec"])
print()
print("Best performance (lowest val loss):", best_val["model"], f"({best_val['val_loss']:.4f})")
print("Best efficiency (highest tok/s):", fastest["model"], f"({fastest['tok_per_sec']:.1f})")

Model                    Params   TrainLoss    ValLoss   Time(s)       Tok/s
--------------------------------------------------------------------------------
Seq2Seq LSTM          5,148,384      9.3722     9.3438     2.813      2093.2
LSTM + Attention      5,247,328      9.3837     9.3387     3.082      1910.2
Transformer           5,546,208      9.1441     8.3584     4.384      1343.2

Best performance (lowest val loss): Transformer (8.3584)
Best efficiency (highest tok/s): Seq2Seq LSTM (2093.2)
